In [ ]:
# Install required packages first
!pip install -q transformers accelerate>=1.0.0 datasets>=3.0.0
!pip install -q av librosa soundfile resampy
!pip install -q scikit-learn matplotlib seaborn
!pip uninstall -y -q torch torchvision torchaudio
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q pytorchvideo
!pip install -q evaluate
!pip install -q datasets

zsh:1: 1.0.0 not found
^C
ERROR: Could not find a version that satisfies the requirement torch (from versions: none)
ERROR: No matching distribution found for torch


In [ ]:
# ===== Fix logging + tqdm =====
from transformers.utils import logging
logging.set_verbosity_info()
logging.enable_progress_bar()

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import os
import gc
import cv2
import warnings
from typing import Dict, List, Optional, Tuple, Union
from sklearn.metrics import accuracy_score, classification_report, f1_score, recall_score, precision_score
from dataclasses import dataclass
import time
import subprocess
import sys
from contextlib import contextmanager, redirect_stderr
import io
from transformers.trainer_utils import get_last_checkpoint
# Transformers components
from transformers import (
    AutoTokenizer, AutoModel, AutoImageProcessor,
    TimesformerForVideoClassification, Trainer, TrainingArguments, 
    EvalPrediction, PreTrainedModel, PretrainedConfig
)
from torch.utils.data import Dataset, DataLoader
import librosa

/Users/nnam/miniconda3/envs/mtik/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from typing import Literal
def resolve_path(relative_path, mode:Literal['kaggle','colab','local']):
    """
    Tự động map path theo môi trường (kaggle / colab / local)
    """
    if relative_path=='' or relative_path==None or pd.isna(relative_path):
        print(f'File không tồn tại: {relative_path}')
        return None
    
    # Kaggle
    if mode == 'kaggle':
        base_dirs = [
            "/kaggle/input",
            "/kaggle/working"
        ]
    
    # Colab
    elif mode == 'colab':
        base_dirs = [
            "/content",
            "/content/drive/MyDrive"
        ]
    
    # Local
    elif mode == 'local':
        base_dirs = [
            "./datasets",
            "."
        ]
    
    # tìm path tồn tại
    for base in base_dirs:
        full_path = os.path.join(str(base), str(relative_path))
        if os.path.exists(full_path):
            return full_path
    print(f"Không tìm thấy: {full_path}")
    return full_path

### Validator

In [16]:
# =============================================================================
# COMPLETE ERROR SUPPRESSION AND ENVIRONMENT SETUP
# =============================================================================

def setup_ultimate_silent_environment():
    """Setup the most comprehensive silent environment"""
    
    # Environment variables for complete silence
    silent_vars = {
        'OPENCV_LOG_LEVEL': 'SILENT',
        'OPENCV_VIDEOIO_DEBUG': '0',
        'OPENCV_FFMPEG_DEBUG': '0',
        'OPENCV_VIDEOWRITER_DEBUG': '0',
        'OPENCV_VIDEOCAPTURE_DEBUG': '0',
        'OPENCV_FFMPEG_LOGLEVEL': '-8',
        'OPENCV_AVFOUNDATION_SKIP_AUTH': '1',
        'FFREPORT': 'file=/dev/null:level=quiet',
        'WANDB_DISABLED': 'true',
        'TOKENIZERS_PARALLELISM': 'false',
        'PYTHONWARNINGS': 'ignore',
        'CUDA_LAUNCH_BLOCKING': '0'
    }
    
    for key, value in silent_vars.items():
        os.environ[key] = value
    
    # Configure OpenCV for complete silence
    try:
        cv2.setLogLevel(0)
        cv2.setUseOptimized(True)
    except:
        pass
    
    # Suppress all warnings
    warnings.filterwarnings('ignore')
    warnings.simplefilter('ignore')

@contextmanager
def ultimate_stderr_suppression():
    """Most advanced stderr suppression for C libraries"""
    old_stderr = None
    try:
        if hasattr(os, 'devnull'):
            old_stderr = sys.stderr
            sys.stderr = open(os.devnull, 'w')
            yield
        else:
            old_stderr = sys.stderr
            sys.stderr = io.StringIO()
            yield
    except:
        yield
    finally:
        if old_stderr is not None:
            try:
                if hasattr(sys.stderr, 'close'):
                    sys.stderr.close()
            except:
                pass
            sys.stderr = old_stderr

# Setup environment immediately
setup_ultimate_silent_environment()
print("=== SMART MULTIMODAL CLASSIFIER - ADAPTIVE VALIDATION (FIXED) ===")

def cleanup_memory():
    """Enhanced memory cleanup"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

cleanup_memory()

# =============================================================================
# 1. SMART VIDEO VALIDATION SYSTEM (SAME AS BEFORE)
# =============================================================================

import os
import subprocess
from contextlib import contextmanager
from typing import Optional

import cv2
import pandas as pd
import torch
import torchaudio


class DatasetValidator:
    """validator for both video and audio with separate stats/caches."""

    def __init__(self, strict_mode: bool = False):
        self.strict_mode = strict_mode

        # Separate caches
        self.video_cache = {}
        self.audio_cache = {}

        # Availability
        self.ffprobe_available = None
        self._check_ffprobe_availability()

        # Separate stats
        self.video_stats = {
            "total": 0,
            "valid": 0,
            "invalid": 0,
            "not_found": 0,
            "too_small": 0,
            "ffprobe_success": 0,
            "ffprobe_failed": 0,
            "opencv_success": 0,
            "opencv_failed": 0,
            "decode_failed": 0,
        }

        self.audio_stats = {
            "total": 0,
            "valid": 0,
            "invalid": 0,
            "not_found": 0,
            "too_small": 0,
            "load_success": 0,
            "load_failed": 0,
            "too_short": 0,
            "silent": 0,
            "nan_inf": 0,
        }

        print("📋 Validator initialized:")
        print(f"  FFprobe available: {self.ffprobe_available}")
        print(f"  Strict mode: {self.strict_mode}")

    def _check_ffprobe_availability(self):
        """Check if FFprobe is available on the system."""
        try:
            with self._suppress_stderr():
                result = subprocess.run(
                    ["ffprobe", "-version"],
                    capture_output=True,
                    timeout=5,
                )
            self.ffprobe_available = (result.returncode == 0)
        except:
            self.ffprobe_available = False

        if not self.ffprobe_available:
            print("⚠️ FFprobe not available - will use OpenCV-only validation")

    @contextmanager
    def _suppress_stderr(self):
        """Fallback stderr suppression if you don't already have ultimate_stderr_suppression()."""
        try:
            import sys
            old_stderr = sys.stderr
            sys.stderr = open(os.devnull, "w")
            yield
        finally:
            try:
                sys.stderr.close()
            except:
                pass
            sys.stderr = old_stderr

    # =========================================================
    # VIDEO VALIDATION
    # =========================================================

    def is_video_valid(self, video_path: str) -> bool:
        """Validate a single video file."""
        if video_path in self.video_cache:
            return self.video_cache[video_path]

        self.video_stats["total"] += 1
        is_valid = False

        try:
            if not video_path or not os.path.exists(video_path):
                self.video_stats["not_found"] += 1
                self.video_cache[video_path] = False
                return False

            try:
                file_size = os.path.getsize(video_path)
                if file_size < 1000:
                    self.video_stats["too_small"] += 1
                    self.video_cache[video_path] = False
                    return False
            except:
                self.video_stats["too_small"] += 1
                self.video_cache[video_path] = False
                return False

            if self.ffprobe_available:
                if self._validate_video_with_ffprobe(video_path):
                    is_valid = True
                    self.video_stats["ffprobe_success"] += 1
                else:
                    self.video_stats["ffprobe_failed"] += 1
                    if self._validate_video_with_opencv(video_path):
                        is_valid = True
                        self.video_stats["opencv_success"] += 1
                    else:
                        self.video_stats["opencv_failed"] += 1
                        self.video_stats["decode_failed"] += 1
            else:
                if self._validate_video_with_opencv(video_path):
                    is_valid = True
                    self.video_stats["opencv_success"] += 1
                else:
                    self.video_stats["opencv_failed"] += 1
                    self.video_stats["decode_failed"] += 1

        except:
            self.video_stats["invalid"] += 1

        if is_valid:
            self.video_stats["valid"] += 1
        else:
            self.video_stats["invalid"] += 1

        self.video_cache[video_path] = is_valid
        return is_valid

    def _validate_video_with_ffprobe(self, video_path: str) -> bool:
        """Very lenient FFprobe-based validation."""
        try:
            cmd = [
                "ffprobe", "-v", "quiet",
                "-select_streams", "v:0",
                "-show_entries", "stream=width,height,nb_frames",
                "-of", "csv=p=0",
                video_path
            ]

            with self._suppress_stderr():
                result = subprocess.run(cmd, capture_output=True, text=True, timeout=10)

            if result.returncode != 0:
                return False

            output = result.stdout.strip()
            if not output:
                return False

            parts = output.split(",")
            if len(parts) >= 2:
                width = int(parts[0]) if parts[0] else 0
                height = int(parts[1]) if parts[1] else 0
                return width > 0 and height > 0

            return False

        except:
            return False

    def _validate_video_with_opencv(self, video_path: str) -> bool:
        """Very lenient OpenCV-based validation."""
        cap = None
        try:
            with self._suppress_stderr():
                for backend in [cv2.CAP_FFMPEG, cv2.CAP_ANY]:
                    try:
                        cap = cv2.VideoCapture(video_path, backend)
                        if cap and cap.isOpened():
                            break
                    except:
                        continue

                if not cap or not cap.isOpened():
                    return False

                width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

                if width > 0 and height > 0:
                    ret, frame = cap.read()
                    if ret and frame is not None and frame.size > 0:
                        return True

                return False

        except:
            return False
        finally:
            if cap:
                try:
                    cap.release()
                except:
                    pass

    # =========================================================
    # AUDIO VALIDATION
    # =========================================================

    def is_audio_valid(self, audio_path: str, min_duration_sec: float = 0.1) -> bool:
        """Validate a single audio file."""
        if audio_path in self.audio_cache:
            return self.audio_cache[audio_path]

        self.audio_stats["total"] += 1
        is_valid = False

        try:
            if not audio_path or not os.path.exists(audio_path):
                self.audio_stats["not_found"] += 1
                self.audio_cache[audio_path] = False
                return False

            try:
                file_size = os.path.getsize(audio_path)
                if file_size < 1000:
                    self.audio_stats["too_small"] += 1
                    print(f'Too small video: {audio_path}')
                    self.audio_cache[audio_path] = False
                    return False
            except:
                self.audio_stats["too_small"] += 1
                print(f'Too small video: {audio_path}')
                self.audio_cache[audio_path] = False
                return False

            try:
                waveform, sr = librosa.load(audio_path, mono=True, sr=48000) 

                if waveform is None:
                    self.audio_stats["load_failed"] += 1
                    self.audio_cache[audio_path] = False
                    return False
                if waveform.size == 0:
                    self.audio_stats["invalid"] += 1
                    self.audio_cache[audio_path] = False
                    return False
                # duration
                duration = len(waveform) / float(sr) if sr and sr > 0 else 0.0
                if duration < min_duration_sec:
                    self.audio_stats["too_short"] += 1
                    self.audio_cache[audio_path] = False
                    return False

                # NaN / Inf
                if np.isnan(waveform).any() or np.isinf(waveform).any():
                    self.audio_stats["nan_inf"] += 1
                    self.audio_cache[audio_path] = False
                    return False

                # silent
                if np.mean(np.abs(waveform)) < 1e-6:
                    self.audio_stats["silent"] += 1
                    print(f'Silent video: {audio_path}')
                    self.audio_cache[audio_path] = False
                    return False
                is_valid = True
                self.audio_stats["load_success"] += 1

            except:
                self.audio_stats["load_failed"] += 1
                is_valid = False

        except:
            self.audio_stats["invalid"] += 1

        if is_valid:
            self.audio_stats["valid"] += 1
        else:
            self.audio_stats["invalid"] += 1

        self.audio_cache[audio_path] = is_valid
        return is_valid

    # =========================================================
    # DATASET VALIDATION
    # =========================================================

    def validate_dataset_sample(
        self,
        df: pd.DataFrame,
        sample_size: int = 100,
        video_col: str = "video_path",
        audio_col: str = "audio_path",
        mode: Literal['local', 'colab', 'kaggle'] = 'local'
    ) -> pd.DataFrame:
        """
        Validate BOTH video + audio together
        A sample is valid only if BOTH are valid
        """

        print(f"🧪 Testing multimodal (video + audio) validation on {sample_size} samples...")

        sample_df = df.head(sample_size) if len(df) > sample_size else df
        valid_count = 0

        for _, row in sample_df.iterrows():
            video_path = resolve_path(row.get(video_col, None), mode=mode)
            audio_path = resolve_path(row.get(audio_col, None), mode=mode)

            video_ok = self.is_video_valid(video_path)
            audio_ok = self.is_audio_valid(audio_path)

            if video_ok and audio_ok:
                valid_count += 1

        success_rate = (valid_count / len(sample_df)) * 100 if len(sample_df) > 0 else 0.0

        print(f"📊 Multimodal validation results:")
        print(f"  Tested: {len(sample_df)}")
        print(f"  Valid (both audio + video): {valid_count}")
        print(f"  Success rate: {success_rate:.1f}%")

        if success_rate < 10:
            print("⚠️ Very low success rate!")
            print("🔄 Switching to lenient validation...")
            return self._validate_dataset_lenient(df, video_col, audio_col, mode)
        else:
            return self._validate_dataset_normal(df, video_col, audio_col, mode)

    def _validate_dataset_normal(
        self,
        df: pd.DataFrame,
        video_col: str,
        audio_col: str,
        mode: str
    ) -> pd.DataFrame:

        print("🔍 Performing normal multimodal validation...")

        valid_indices = []
        total = len(df)

        for idx, row in df.iterrows():
            video_path = resolve_path(row.get(video_col, None), mode=mode)
            audio_path = resolve_path(row.get(audio_col, None), mode=mode)

            video_ok = self.is_video_valid(video_path)
            audio_ok = self.is_audio_valid(audio_path)

            if video_ok and audio_ok:
                valid_indices.append(idx)

            if (idx + 1) % 500 == 0:
                print(f"  Progress: {idx + 1}/{total}")

        return df.iloc[valid_indices].reset_index(drop=True)

    def _validate_dataset_lenient(
        self,
        df: pd.DataFrame,
        video_col: str,
        audio_col: str,
        mode: str
    ) -> pd.DataFrame:

        print("🔄 Performing lenient multimodal validation...")

        valid_indices = []
        total = len(df)

        for idx, row in df.iterrows():
            video_path = resolve_path(row.get(video_col, None), mode=mode)
            audio_path = resolve_path(row.get(audio_col, None), mode=mode)

            video_ok = (
                video_path is not None and
                os.path.exists(video_path) and
                os.path.getsize(video_path) > 1000 and
                self._basic_opencv_check(video_path)
            )

            audio_ok = (
                audio_path is not None and
                os.path.exists(audio_path) and
                os.path.getsize(audio_path) > 1000 and
                self._basic_audio_check(audio_path)
            )

            if video_ok and audio_ok:
                valid_indices.append(idx)
                self.video_stats["valid"] += 1
                self.audio_stats["valid"] += 1
            else:
                self.video_stats["invalid"] += 1
                self.audio_stats["invalid"] += 1

            if (idx + 1) % 500 == 0:
                print(f"  Progress: {idx + 1}/{total}")

        return df.iloc[valid_indices].reset_index(drop=True)

    def _basic_opencv_check(self, video_path: str) -> bool:
        """Very basic video open check."""
        try:
            with self._suppress_stderr():
                cap = cv2.VideoCapture(video_path)
                is_opened = cap.isOpened()
                cap.release()
                return is_opened
        except:
            return False

    def _basic_audio_check(self, audio_path: str) -> bool:
        """Very basic audio open check."""
        try:
            waveform, sr = torchaudio.load(audio_path)
            return waveform is not None and waveform.numel() > 0
        except:
            return False

    # =========================================================
    # STATS
    # =========================================================

    def print_stats(self):
        """Print separate video and audio validation statistics."""

        print("\n📊 VIDEO validation statistics:")
        print(f"  Total files: {self.video_stats['total']}")
        print(f"  ✅ Valid files: {self.video_stats['valid']}")
        print(f"  ❌ Invalid files: {self.video_stats['invalid']}")
        print(f"  📁 Not found: {self.video_stats['not_found']}")
        print(f"  📏 Too small: {self.video_stats['too_small']}")
        print(f"  🔧 FFprobe success: {self.video_stats['ffprobe_success']}")
        print(f"  🔧 FFprobe failed: {self.video_stats['ffprobe_failed']}")
        print(f"  📹 OpenCV success: {self.video_stats['opencv_success']}")
        print(f"  📹 OpenCV failed: {self.video_stats['opencv_failed']}")
        print(f"  🚫 Decode failed: {self.video_stats['decode_failed']}")

        print("\n📊 AUDIO validation statistics:")
        print(f"  Total files: {self.audio_stats['total']}")
        print(f"  ✅ Valid files: {self.audio_stats['valid']}")
        print(f"  ❌ Invalid files: {self.audio_stats['invalid']}")
        print(f"  📁 Not found: {self.audio_stats['not_found']}")
        print(f"  📏 Too small: {self.audio_stats['too_small']}")
        print(f"  🎧 Load success: {self.audio_stats['load_success']}")
        print(f"  🎧 Load failed: {self.audio_stats['load_failed']}")
        print(f"  ⏱️ Too short: {self.audio_stats['too_short']}")
        print(f"  🔇 Silent: {self.audio_stats['silent']}")
        print(f"  ⚠️ NaN/Inf: {self.audio_stats['nan_inf']}")

=== SMART MULTIMODAL CLASSIFIER - ADAPTIVE VALIDATION (FIXED) ===


### Data loader

In [ ]:
import torchaudio
from abc import ABC, abstractmethod
from transformers import ClapProcessor
import librosa

class RobustLoader(ABC):
    """Abstract base class for robust media loaders"""

    def __init__(self):
        self.successful_loads = 0
        self.failed_loads = 0

    @abstractmethod
    def load(self, path: str):
        pass

    def _get_fallback(self):
        pass

    def get_stats(self):
        total = self.successful_loads + self.failed_loads
        if total > 0:
            print(f"Loading: {self.successful_loads}/{total} successful "
                  f"({100 * self.successful_loads / total:.1f}%)")

    
class RobustVideoLoader(RobustLoader):
    """Robust video loader with better error handling"""

    def __init__(self, max_frames=8, image_size=224):
        super().__init__()
        self.max_frames = max_frames
        self.image_size = image_size
        self.video_mean = [0.485, 0.456, 0.406]
        self.video_std  = [0.229, 0.224, 0.225]
        self.fallback_videos = self._create_fallback_videos()

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _normalize_video(self, video_tensor: torch.Tensor) -> torch.Tensor:
        """Apply ImageNet normalization (T, C, H, W)"""
        mean = torch.tensor(self.video_mean).view(1, 3, 1, 1)
        std  = torch.tensor(self.video_std).view(1, 3, 1, 1)
        return (video_tensor - mean) / std

    def _create_fallback_videos(self) -> dict:
        """Create multiple types of fallback videos"""
        T, S = self.max_frames, self.image_size
        fallbacks = {}

        # Black video
        fallbacks['black'] = self._normalize_video(
            torch.zeros(T, 3, S, S)
        )

        # Noise video
        fallbacks['noise'] = self._normalize_video(
            torch.randn(T, 3, S, S) * 0.2
        )

        # Gradient pattern
        pattern = torch.zeros(T, 3, S, S)
        x = torch.arange(S).float() / S
        y = torch.arange(S).float() / S
        xx, yy = torch.meshgrid(x, y, indexing='ij')
        for t in range(T):
            for c in range(3):
                pattern[t, c] = (xx + yy + t / T) % 1
        fallbacks['pattern'] = self._normalize_video(pattern)

        return fallbacks

    def _make_dummy_frame(self) -> np.ndarray:
        """Return a normalized black frame (H, W, C)"""
        dummy = np.zeros((self.image_size, self.image_size, 3), dtype=np.float32)
        mean  = np.array(self.video_mean, dtype=np.float32).reshape(1, 1, 3)
        std   = np.array(self.video_std,  dtype=np.float32).reshape(1, 1, 3)
        return (dummy - mean) / std

    def _normalize_frame(self, frame: np.ndarray) -> np.ndarray:
        """Normalize a single HWC float32 frame"""
        mean = np.array(self.video_mean, dtype=np.float32).reshape(1, 1, 3)
        std  = np.array(self.video_std,  dtype=np.float32).reshape(1, 1, 3)
        return (frame - mean) / std

    def _process_raw_frame(self, frame: np.ndarray) -> np.ndarray:
        """Resize, convert color space and normalize a raw BGR frame"""
        frame = cv2.resize(frame, (self.image_size, self.image_size))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = frame.astype(np.float32) / 255.0
        return self._normalize_frame(frame)

    # ------------------------------------------------------------------
    # Fallback
    # ------------------------------------------------------------------

    def _get_fallback(self) -> torch.Tensor:
        """Return a diverse fallback video tensor"""
        keys = ['black', 'noise', 'pattern']
        key  = keys[self.failed_loads % len(keys)]
        return self.fallback_videos[key].clone()

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def load(self, path: str) -> torch.Tensor:
        """Load video with comprehensive error handling → (T, C, H, W)"""
        try:
            cap = None
            for backend in [cv2.CAP_FFMPEG, cv2.CAP_ANY]:
                try:
                    cap = cv2.VideoCapture(path, backend)
                    if cap and cap.isOpened():
                        break
                except Exception:
                    continue

            if not cap or not cap.isOpened():
                self.failed_loads += 1
                return self._get_fallback()

            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 30

            if total_frames <= self.max_frames:
                frame_indices = list(range(total_frames)) + \
                                [total_frames - 1] * (self.max_frames - total_frames)
            else:
                frame_indices = np.linspace(
                    0, total_frames - 1, self.max_frames, dtype=int
                )

            frames = []
            for idx in frame_indices:
                try:
                    cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
                    ret, raw = cap.read()
                    if ret and raw is not None and raw.size > 0:
                        frames.append(self._process_raw_frame(raw))
                    else:
                        frames.append(frames[-1] if frames else self._make_dummy_frame())
                except Exception:
                    frames.append(frames[-1] if frames else self._make_dummy_frame())

            cap.release()

            # Pad if necessary
            while len(frames) < self.max_frames:
                frames.append(frames[-1] if frames else self._make_dummy_frame())

            # (T, H, W, C) → (T, C, H, W)
            video_tensor = torch.from_numpy(
                np.stack(frames[:self.max_frames])
            ).float().permute(0, 3, 1, 2)

            self.successful_loads += 1
            return video_tensor

        except Exception:
            self.failed_loads += 1
            return self._get_fallback()


class RobustAudioLoader(RobustLoader):
    """Audio loader + CLAP preprocessing with fixed 10s length"""

    def __init__(self, target_sr=48000, max_duration=10.0):
        super().__init__()
        self.target_sr = target_sr
        self.max_samples = int(target_sr * max_duration)

        self.processor = ClapProcessor.from_pretrained("laion/larger_clap_general")

    # ------------------------------------------------------------------
    # Fallback
    # ------------------------------------------------------------------

    def _get_fallback(self) -> torch.Tensor:
        return torch.zeros(64, 1000, dtype=torch.float32)

    # ------------------------------------------------------------------
    # Fix length
    # ------------------------------------------------------------------

    def _fix_length(self, waveform: np.ndarray) -> np.ndarray:
        length = len(waveform)

        # 🔥 truncate
        if length > self.max_samples:
            return waveform[:self.max_samples]

        # 🔥 pad
        if length < self.max_samples:
            pad = np.zeros(self.max_samples - length, dtype=np.float32)
            return np.concatenate([waveform, pad])

        return waveform

    # ------------------------------------------------------------------
    # Load waveform
    # ------------------------------------------------------------------

    def _load_waveform(self, path: str) -> np.ndarray:
        waveform, sr = librosa.load(path, sr=self.target_sr, mono=True)  

        if sr != self.target_sr:
            waveform = librosa.resample(waveform, orig_sr=sr, target_sr=self.target_sr)

        waveform = self._fix_length(waveform)

        return waveform.astype(np.float32)

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def load(self, path: str) -> torch.Tensor:
        try:
            waveform = self._load_waveform(path)

            inputs = self.processor(
                audio=waveform,
                sampling_rate=self.target_sr,
                return_tensors="pt"
            )

            features = inputs["input_features"].squeeze(0)  # (64, T)

            self.successful_loads += 1
            return features

        except Exception as e:
            print(f"⚠️ Audio error: {e}")
            self.failed_loads += 1
            return self._get_fallback()

### Data manager

In [ ]:
# =============================================================================
# 3-8. MODEL AND TRAINING CODE (SAME AS BEFORE)
# =============================================================================
import pickle
from transformers import ClapProcessor

class SmartDataManager:
    def __init__(self, csv_path: str):
        self.csv_path = csv_path
        self.df = None
        self.classes = None
        self.class_to_id = None
        self.id_to_class = None
        self.num_classes = None
        
    def load_data(self, mode:Literal['local','colab','kaggle']):
        self.mode = mode
        print("📂 Loading dataset...")
        self.df = pd.read_csv(self.csv_path).dropna(subset=['class_name'])
        
        # KAGGLE_VIDEO_ROOT = "/kaggle/input/datasets/anhoangvo/tikharm-dataset/Dataset"

        # def fix_kaggle_path(old_path):
        #     if not isinstance(old_path, str):
        #         return old_path
            
        #     # Lấy phần sau "Dataset/"
        #     if "Dataset/" in old_path:
        #         relative_part = old_path.split("Dataset/")[-1]
        #         return os.path.join(KAGGLE_VIDEO_ROOT, relative_part)
            
        #     return old_path
        
        # self.df["video_path"] = self.df["video_path"].apply(fix_kaggle_path)

        
        # Fix video paths
        # self.df['video_path'] = self.df['video_path'].str.replace(
        #     r'^/kaggle/working/extra_tikharm/',
        #     '/kaggle/input/extra-dataset/',
        #     regex=True
        # )
        
        self.df['combined_text'] = self.df['combined_text'].fillna("")
        
        print(f"Dataset loaded: {len(self.df)} samples")
        print(f"Classes distribution:\n{self.df['class_name'].value_counts()}")
        self.classes = sorted(self.df['class_name'].unique())
        self.class_to_id = {cls: i for i, cls in enumerate(self.classes)}
        self.id_to_class = {i: cls for cls, i in self.class_to_id.items()}
        self.num_classes = len(self.classes)
        
        print(f"📊 Classes ({self.num_classes}): {self.classes}")
        
        # Print some sample paths to debug
        print(f"\n🔍 Sample video paths:")
        for i in range(min(5, len(self.df))):
            path = resolve_path(self.df.iloc[i]['video_path'], mode=mode)
            exists = os.path.exists(path)
            print(f"  {path} - Exists: {exists}")
            
        # Print some sample paths to debug
        print(f"\n🔍 Sample audio paths:")
        for i in range(min(5, len(self.df))):
            path = resolve_path(self.df.iloc[i]['audio_path'], mode=mode)
            exists = os.path.exists(path)
            print(f"  {path} - Exists: {exists}")
        
    def get_smart_splits(self, validator: DatasetValidator):
        """Get splits with smart validation"""
        print("🧹 Creating smart validated splits...")
        
        # Get original splits
        train_df = self.df[self.df['split'] == 'train'].reset_index(drop=True)
        val_df = self.df[self.df['split'] == 'val'].reset_index(drop=True)
        test_df = self.df[self.df['split'] == 'test'].reset_index(drop=True)

        # train_df = self.df[(self.df['split'] == 'train') & (self.df['original_dir'] != 'extra')].reset_index(drop=True)
        # val_df = self.df[(self.df['split'] == 'val') & (self.df['original_dir'] != 'extra')].reset_index(drop=True)
        # test_df = self.df[self.df['split'] == 'test'].reset_index(drop=True)
        
        print(f"Original splits - Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
        
        # Use sample validation first
        print('Validating Audio ...')
        print("     Validating training set...")
        clean_train_df = validator.validate_dataset_sample(train_df, sample_size=100, mode=self.mode)
        if len(clean_train_df) < 50:
            print("⚠️ Too few training samples, using lenient mode for all...")
            clean_train_df = validator._validate_dataset_lenient(train_df, video_col='video_path', audio_col='audio_path', mode=self.mode)
        
        print("     Validating validation set...")
        clean_val_df = validator.validate_dataset_sample(val_df, sample_size=50, mode=self.mode)
        if len(clean_val_df) < 10:
            clean_val_df = validator._validate_dataset_lenient(val_df, video_col='video_path', audio_col='audio_path', mode=self.mode)
        
        print("     Validating test set...")
        clean_test_df = validator.validate_dataset_sample(test_df, sample_size=50, mode=self.mode)
        if len(clean_test_df) < 10:
            clean_test_df = validator._validate_dataset_lenient(test_df, video_col='video_path', audio_col='audio_path', mode=self.mode)
        
        validator.print_stats()
        
        print(f"\n✨ Final splits:")
        print(f"  🏋️ Train: {len(clean_train_df)} (was {len(train_df)})")
        print(f"  ✅ Val: {len(clean_val_df)} (was {len(val_df)})")
        print(f"  🧪 Test: {len(clean_test_df)} (was {len(test_df)})")
        
        return clean_train_df, clean_val_df, clean_test_df

class VideoAudioDataset(Dataset):
    """Smart dataset with robust video loading AND TRACKING"""
    
    def __init__(self, dataframe, max_frames=8, image_size=224, 
                 dataset_name="unknown", target_sr=48000, max_duration=10, mode=Literal['colab', 'kaggle', 'local']
                ):
        
        self.df = dataframe
        self.mode=mode
        # self.hashtag_vector_map = hashtag_vector_map
        # self.text_tokenizer = text_tokenizer
        self.max_frames = max_frames
        self.image_size = image_size
        self.dataset_name = dataset_name  # Track dataset name
        
        # FIXED: Store actual dataframe length at creation
        self.actual_length = len(self.df)
        
        self.video_loader = RobustVideoLoader(max_frames, image_size)
        self.audio_loader = RobustAudioLoader(target_sr=target_sr, max_duration=max_duration)
 
        print(f"🛡️ {self.dataset_name} dataset: {self.actual_length} samples (verified)")
        
        # Store indices for verification
        self.sample_indices = list(self.df.index)
        
    def __len__(self):
        # FIXED: Return actual dataframe length, not processed count
        return self.actual_length
    
    def __getitem__(self, idx):
        try:
            row = self.df.iloc[idx]
            video_path = resolve_path(row['video_path'], mode=self.mode)
            audio_path = resolve_path(row['audio_path'], mode=self.mode)

            # Load video 
            pixel_values = self.video_loader.load(video_path)
            
            # Get label
            label = data_manager.class_to_id[row['class_name']]

            if audio_path:
                audio_input_features = self.audio_loader.load(audio_path)
            else:
                audio_input_features = torch.zeros(64, 1000, dtype=torch.float32)

            return {
                'pixel_values': pixel_values,
                'audio_input_features': audio_input_features,
                'audio_is_longer': torch.tensor(False),
                'labels': torch.tensor(label, dtype=torch.long)
            }
            
        except Exception as e:
            print(f"⚠️ Error at index {idx} in {self.dataset_name}: {e}")
            
            # Safe fallback
            dummy_video = torch.zeros(self.max_frames, 3, self.image_size, self.image_size)
            dummy_audio_vec = torch.zeros(64, 1000, dtype=torch.float32)
            
            return {
                'pixel_values': dummy_video,
                'audio_input_features': dummy_audio_vec,
                'audio_is_longer': torch.tensor(False),
                'labels': torch.tensor(0, dtype=torch.long)
            }
    
    def get_stats(self):
        print(f"📊 {self.dataset_name} stats:")
        print(f"  DataFrame length: {len(self.df)}")
        print(f"  Actual length: {self.actual_length}")
        self.video_loader.get_stats()

def compute_comprehensive_metrics(eval_pred: EvalPrediction) -> Dict[str, float]:
    """Compute all 4 key metrics"""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    labels = eval_pred.label_ids
    
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro', zero_division=0)
    recall = recall_score(labels, predictions, average='macro', zero_division=0)
    precision = precision_score(labels, predictions, average='macro', zero_division=0)
    
    return {
        'accuracy': accuracy,
        'f1_score': f1,
        'recall': recall,
        'precision': precision
    }
        
def smart_collate_fn(batch):
    """Smart collate function"""
    try:
        return {
                'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
                'audio_input_features': torch.stack([x['audio_input_features'] for x in batch]),
                'audio_is_longer': torch.stack([x['audio_is_longer'] for x in batch]),
                'labels': torch.stack([x['labels'] for x in batch]),
            }

    except Exception as e:
        print(f"⚠️ Collate error: {e}")
        batch_size = len(batch)

        return {
            'pixel_values': torch.zeros(batch_size, 8, 3, 224, 224),
            'audio_input_features': torch.zeros(batch_size, 64, 1000),
            'audio_is_longer': torch.zeros(batch_size, dtype=torch.bool),
            'labels': torch.zeros(batch_size, dtype=torch.long)
        }

### Model architecture

In [ ]:
from dataclasses import dataclass
from transformers import PretrainedConfig, PreTrainedModel
from transformers import TimesformerForVideoClassification, ClapAudioModelWithProjection
import torch
from torch import nn
import torch.nn.functional as F

@dataclass
class SmartMultimodalConfig(PretrainedConfig):
    model_type = "smart_multimodal"

    def __init__(
        self,
        num_classes: int = 4,
        video_model_name: str = "facebook/timesformer-base-finetuned-k400",
        audio_model_name: str = "laion/larger_clap_general",
        video_hidden_size: int = 768,
        audio_hidden_size: int = 512,
        fusion_hidden_size: int = 256,
        classifier_hidden_size: int = 128,
        max_frames: int = 8,
        image_size: int = 224,
        dropout: float = 0.1,
        freeze_backbone: bool = False,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.video_model_name = video_model_name
        self.audio_model_name = audio_model_name
        self.video_hidden_size = video_hidden_size
        self.audio_hidden_size = audio_hidden_size
        self.fusion_hidden_size = fusion_hidden_size
        self.classifier_hidden_size = classifier_hidden_size
        self.max_frames = max_frames
        self.image_size = image_size
        self.dropout = dropout
        self.freeze_backbone = freeze_backbone


class SmartMultimodalModel(PreTrainedModel):
    config_class = SmartMultimodalConfig

    def __init__(self, config):
        super().__init__(config)
        self.config = config
        print(f"🎬 Initializing video model: {config.video_model_name}")
        self.video_encoder = TimesformerForVideoClassification.from_pretrained(
            config.video_model_name,
            num_labels=config.num_classes,
            ignore_mismatched_sizes=True
        )

        print(f"🎧 Initializing audio model: {config.audio_model_name}")
        self.audio_encoder = ClapAudioModelWithProjection.from_pretrained(
            config.audio_model_name
        )

        # Freeze backbones
        if config.freeze_backbone:
            for param in self.video_encoder.parameters():
                param.requires_grad = False
            for param in self.audio_encoder.parameters():
                param.requires_grad = False

            print("Backbones are frozen.")
        else:
            print("Train full pipeline")
        # Dimensions
        video_dim = self.video_encoder.config.hidden_size
        audio_dim = getattr(self.audio_encoder.config, "projection_dim", config.audio_hidden_size)

        self.config.video_hidden_size = video_dim
        self.config.audio_hidden_size = audio_dim

        print(f"📐 Model dimensions - Video: {video_dim}, Audio: {audio_dim}")

        # Projections
        self.video_projection = nn.Sequential(
            nn.Linear(video_dim, config.fusion_hidden_size),
            nn.ReLU(),
            nn.Dropout(config.dropout)
        )

        # CLAP output is usually 512-d, project to fusion space
        self.audio_projection = nn.Sequential(
            nn.Linear(audio_dim, config.fusion_hidden_size),
            nn.ReLU(),
            nn.Dropout(config.dropout)
        )

        self.fusion = nn.Sequential(
            nn.Linear(config.fusion_hidden_size * 2, config.fusion_hidden_size),
            nn.LayerNorm(config.fusion_hidden_size),
            nn.ReLU(),
            nn.Dropout(config.dropout),

            nn.Linear(config.fusion_hidden_size, config.classifier_hidden_size),
            nn.LayerNorm(config.classifier_hidden_size),
            nn.ReLU(),
            nn.Dropout(config.dropout)
        )

        self.classifier = nn.Linear(config.classifier_hidden_size, config.num_classes)

        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"🧠 Model: {total_params:,} total, {trainable_params:,} trainable")

    def forward(
        self,
        pixel_values,
        audio_input_features=None,
        audio_is_longer=None,
        labels=None,
        **kwargs
    ):
        device = pixel_values.device
        batch_size = pixel_values.size(0)

        # ===== VIDEO =====
        try:
            video_outputs = self.video_encoder.timesformer(pixel_values)
            video_features = video_outputs.last_hidden_state[:, 0]
        except Exception:
            video_features = torch.zeros(
                batch_size,
                self.config.video_hidden_size,
                device=device
            )

        video_features = self.video_projection(video_features)

        # ===== AUDIO (CLAP ONLY) =====
        try:
            if audio_input_features is not None:
                audio_input_features = audio_input_features.to(device)

                if audio_is_longer is None:
                    audio_is_longer = torch.zeros(batch_size, dtype=torch.bool, device=device)
                else:
                    audio_is_longer = audio_is_longer.to(device)

                audio_outputs = self.audio_encoder(
                    input_features=audio_input_features,
                    is_longer=audio_is_longer
                )

                audio_features = audio_outputs.audio_embeds  # (B, 512)

            else:
                raise ValueError("Missing audio_input_features")

        except Exception:
            # fallback
            audio_features = torch.zeros(
                batch_size,
                self.config.audio_hidden_size,
                device=device
            )

        audio_features = self.audio_projection(audio_features)

        # ===== FUSION =====
        combined = torch.cat(
            [audio_features, video_features],
            dim=1
        )

        fused = self.fusion(combined)
        logits = self.classifier(fused)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(label_smoothing=0.1)
            loss = loss_fct(logits, labels)

        return {"loss": loss, "logits": logits}

### Test

In [14]:
def quick_test_pipeline(dataset, model, device="cuda"):
    print("🚀 Running quick pipeline test...\n")

    # ===== 1. TEST 1 SAMPLE =====
    print("🔹 Test single sample...")
    sample = dataset[0]

    for k, v in sample.items():
        if torch.is_tensor(v):
            print(f"{k}: {v.shape}")
        else:
            print(f"{k}: {type(v)}")

    # ===== 2. TEST BATCH =====
    print("\n🔹 Test batch (collate)...")
    batch = [dataset[i] for i in range(2)]
    batch = smart_collate_fn(batch)

    for k, v in batch.items():
        if torch.is_tensor(v):
            print(f"{k}: {v.shape}")

    # ===== 3. TEST MODEL FORWARD =====
    print("\n🔹 Test model forward...")

    model = model.to(device)
    model.eval()

    with torch.no_grad():
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)

        print("✅ Forward pass success!")
        print(f"logits shape: {outputs['logits'].shape}")

    print("\n🎉 ALL TESTS PASSED!")


In [ ]:
global data_manager
data_manager = SmartDataManager("/Users/nnam/Documents/Workspace/university/mguard/TikTok_Classification/datasets/dataset_with_relative_path.csv")
data_manager.load_data(mode='local')

# Initialize smart validator (not strict)
print("🔍 Initializing smart video validator...")
validator = DatasetValidator(strict_mode=False)

# Get smart validated data
train_df, val_df, test_df = data_manager.get_smart_splits(validator)


# VERIFY DATA SPLITS AGAIN
print(f"\n🔍 VERIFICATION - Actual dataframe sizes:")
print(f"  Train DF: {len(train_df)} rows")
print(f"  Val DF: {len(val_df)} rows") 
print(f"  Test DF: {len(test_df)} rows")

validator.print_stats()

# Check data sufficiency
if len(train_df) < 50:
    print("⚠️ Very few training samples!")
    print("🔄 Proceeding with available data...")
    
    
train_dataset = VideoAudioDataset(
    dataframe=train_df, 
    max_frames=8, 
    image_size=224, 
    target_sr=48000, 
    max_duration=10,
    dataset_name="TRAIN",
    mode='local'    
    )
config = SmartMultimodalConfig(
        num_classes = 4,
        text_model_name = "bert-base-multilingual-cased",
        video_model_name = "facebook/timesformer-base-finetuned-k400",
        audio_model_name = "laion/larger_clap_general",
        video_hidden_size = 768,
        audio_hidden_size = 512,
        fusion_hidden_size = 256,
        classifier_hidden_size = 128,
        max_frames = 8,
        image_size = 224,
        dropout = 0.1,
        freeze_backbone=False
    )
    
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"💻 Using device: {device}")

model = SmartMultimodalModel(config)
model.to(device)

quick_test_pipeline(dataset=train_dataset,model=model, device='cpu')

📂 Loading dataset...
Dataset loaded: 4723 samples
Classes distribution:
class_name
Safe               1248
Harmful Content    1193
Adult Content      1141
Suicide            1141
Name: count, dtype: int64
📊 Classes (4): ['Adult Content', 'Harmful Content', 'Safe', 'Suicide']

🔍 Sample video paths:
  ./datasets/tikharm-dataset/train/Adult Content/jaypark_foryou_7339639860354911493.mp4 - Exists: True
  ./datasets/tikharm-dataset/train/Adult Content/1life.anthonyblane_7368846422709521671.mp4 - Exists: True
  ./datasets/tikharm-dataset/train/Adult Content/loungebarvip_7252268249243454725.mp4 - Exists: True
  ./datasets/tikharm-dataset/train/Adult Content/phangphong111_7287936860796620039.mp4 - Exists: True
  ./datasets/tikharm-dataset/train/Adult Content/girl_xinh.sexy_7298717939052711170.mp4 - Exists: True

🔍 Sample audio paths:
  ./datasets/audios/jaypark_foryou_7339639860354911493.mp3 - Exists: True
  ./datasets/audios/1life.anthonyblane_7368846422709521671.mp3 - Exists: True
  ./datase

Loading weights: 100%|██████████| 249/249 [00:00<00:00, 34082.23it/s]
TimesformerForVideoClassification LOAD REPORT from: facebook/timesformer-base-finetuned-k400
Key               | Status   |                                                                                         
------------------+----------+-----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([400]) vs model:torch.Size([4])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([400, 768]) vs model:torch.Size([4, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


🎧 Initializing audio model: laion/larger_clap_general


Loading weights: 100%|██████████| 348/348 [00:00<00:00, 45539.06it/s]
ClapAudioModelWithProjection LOAD REPORT from: laion/larger_clap_general
Key                                                                 | Status     |  | 
--------------------------------------------------------------------+------------+--+-
text_model.encoder.layer.{0...11}.attention.output.LayerNorm.bias   | UNEXPECTED |  | 
text_model.encoder.layer.{0...11}.intermediate.dense.bias           | UNEXPECTED |  | 
text_model.encoder.layer.{0...11}.output.LayerNorm.bias             | UNEXPECTED |  | 
text_model.encoder.layer.{0...11}.output.LayerNorm.weight           | UNEXPECTED |  | 
text_model.encoder.layer.{0...11}.attention.self.query.bias         | UNEXPECTED |  | 
text_model.encoder.layer.{0...11}.output.dense.weight               | UNEXPECTED |  | 
text_model.encoder.layer.{0...11}.output.dense.bias                 | UNEXPECTED |  | 
text_model.encoder.layer.{0...11}.attention.self.key.bias           | UNEX

Train full pipeline
📐 Model dimensions - Video: 768, Audio: 512
🧠 Model: 190,367,392 total, 190,367,392 trainable
🚀 Running quick pipeline test...

🔹 Test single sample...
pixel_values: torch.Size([8, 3, 224, 224])
audio_input_features: torch.Size([64, 1000])
audio_is_longer: torch.Size([])
labels: torch.Size([])

🔹 Test batch (collate)...
pixel_values: torch.Size([2, 8, 3, 224, 224])
audio_input_features: torch.Size([2, 64, 1000])
audio_is_longer: torch.Size([2])
labels: torch.Size([2])

🔹 Test model forward...
✅ Forward pass success!
logits shape: torch.Size([2, 4])

🎉 ALL TESTS PASSED!


In [ ]:
from torchinfo import summary

summary(
    model,
    input_data={
        "pixel_values": torch.randn(1, 8, 3, 224, 224),
        "audio_input_features": torch.randn(1, 1, 1000, 64),  # ✅ đúng
        "audio_is_longer": torch.tensor([False]),
        "hashtags": torch.randn(1, 4),
    },
    depth=3)

In [ ]:
# from torch.utils.tensorboard import SummaryWriter
# def train(
#     tensorboardpath,
#     checkpointpath
# ):
#     if not os.path.exists(tensorboardpath):
#         os.makedirs(tensorboardpath)
#     if not os.path.exists(checkpointpath):
#         os.makedirs(checkpointpath)
#     writer = SummaryWriter()
    
#     global data_manager
#     data_manager = SmartDataManager("/kaggle/input/datasets/dnnamm/dataset-with-all-hashtag/dataset_with_all_hashtag.csv")
#     data_manager.load_data()
    
#     # Initialize smart validator (not strict)
#     print("🔍 Initializing smart video validator...")
#     validator = SmartVideoValidator(strict_mode=False)
    
#     # Get smart validated data
#     train_df, val_df, test_df = data_manager.get_smart_splits(validator)
    
#     # VERIFY DATA SPLITS AGAIN
#     print(f"\n🔍 VERIFICATION - Actual dataframe sizes:")
#     print(f"  Train DF: {len(train_df)} rows")
#     print(f"  Val DF: {len(val_df)} rows") 
#     print(f"  Test DF: {len(test_df)} rows")
    
#     # Check data sufficiency
#     if len(train_df) < 50:
#         print("⚠️ Very few training samples!")
#         print("🔄 Proceeding with available data...")
    
#     # Initialize models
#     print("🧠 Initializing models...")
#     # text_model_name = "bert-base-multilingual-cased"
#     # text_tokenizer = AutoTokenizer.from_pretrained(text_model_name)
    
#     config = SmartMultimodalConfig(
#         num_classes = 4,
#         text_model_name = "bert-base-multilingual-cased",
#         video_model_name = "facebook/timesformer-base-finetuned-k400",
#         audio_model_name = "laion/larger_clap_general",
#         text_hidden_size = 768,
#         video_hidden_size = 768,
#         audio_hidden_size = 512,
#         fusion_hidden_size = 256,
#         classifier_hidden_size = 128,
#         max_frames = 8,
#         image_size = 224,
#         dropout = 0.1,
#     )
    
#     device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#     print(f"💻 Using device: {device}")
    
#     model = SmartMultimodalModel(config)
#     model.to(device)
    

### Train

In [19]:
def train(
    # Data congig
    data_path:str = "/Users/nnam/Documents/Workspace/university/mguard/TikTok_Classification/datasets/dataset_with_relative_path.csv",
    data_mode:Literal['colab','kaggle', 'local']='local',
    # Video config
    max_frames=8, 
    image_size=224,  
    #Audio config
    target_sr=48000, 
    max_duration=10,
    # model config
    resume_training:bool = False,
    video_model_name = "facebook/timesformer-base-finetuned-k400",
    audio_model_name = "laion/larger_clap_general",
    video_hidden_size = 768,
    audio_hidden_size = 512,
    fusion_hidden_size = 256,
    classifier_hidden_size = 128,
    dropout = 0.1,
    freeze_backbone=False,
    device:Literal['cuda', 'cpu']='cuda',
    #Training config
    checkpoint_dir='',
    best_checkpoint_dir='',
    num_train_epochs = 20,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_score",
    fp16=True,
    dataloader_num_workers=0,
    remove_unused_columns=False,
    save_total_limit = 3,
    dataloader_drop_last=False,
    max_grad_norm=1.0,
    disable_tqdm=False,
    report_to="tensorboard",
    logging_dir="./runs"
):
    
    try:
        print("🚀 Starting smart adaptive training (FIXED VERSION)...")
        
        # Initialize data manager
        global data_manager
        data_manager = SmartDataManager(data_path)
        data_manager.load_data(mode=data_mode)
        
        # Initialize smart validator (not strict)
        print("🔍 Initializing smart video validator...")
        validator = DatasetValidator(strict_mode=False)
        
        # Get smart validated data
        train_df, val_df, test_df = data_manager.get_smart_splits(validator)
        
     
        # VERIFY DATA SPLITS AGAIN
        print(f"\n🔍 VERIFICATION - Actual dataframe sizes:")
        print(f"  Train DF: {len(train_df)} rows")
        print(f"  Val DF: {len(val_df)} rows") 
        print(f"  Test DF: {len(test_df)} rows")
        
        validator.print_stats()
        # Check data sufficiency
        if len(train_df) < 50:
            print("⚠️ Very few training samples!")
            print("🔄 Proceeding with available data...")
        
        # Initialize models
        print("🧠 Initializing models...")
        # text_model_name = "bert-base-multilingual-cased"
        # text_tokenizer = AutoTokenizer.from_pretrained(text_model_name)
        
        config = SmartMultimodalConfig(
            num_classes = data_manager.num_classes,
            video_model_name = video_model_name,
            audio_model_name = audio_model_name,
            video_hidden_size = video_hidden_size,
            audio_hidden_size = audio_hidden_size,
            fusion_hidden_size = fusion_hidden_size,
            classifier_hidden_size = classifier_hidden_size,
            max_frames = max_frames,
            image_size = image_size,
            dropout = 0.1,
            freeze_backbone = freeze_backbone
        )
        
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"💻 Using device: {device}")
        
        model = SmartMultimodalModel(config)
        model.to(device)
        
        # Create smart datasets WITH PROPER TRACKING
        print("📊 Creating smart datasets with tracking...")
        train_dataset = VideoAudioDataset(
            dataframe=train_df, 
            max_frames=max_frames, 
            image_size=image_size, 
            target_sr=target_sr, 
            max_duration=max_duration,
            dataset_name="TRAIN SET",
            mode=data_mode   
        )
        val_dataset = VideoAudioDataset(
            dataframe=val_df, 
            max_frames=max_frames, 
            image_size=image_size, 
            target_sr=target_sr, 
            max_duration=max_duration,
            dataset_name="VAL SET",
            mode=data_mode    
        )
        
        test_dataset = VideoAudioDataset(
            dataframe=test_df, 
            max_frames=max_frames, 
            image_size=image_size, 
            target_sr=target_sr, 
            max_duration=max_duration,
            dataset_name="TEST SET",
            mode=data_mode    
        )
        
        # VERIFY DATASET LENGTHS
        print(f"\n🎯 FINAL DATASET VERIFICATION:")
        print(f"  Train dataset: {len(train_dataset)} samples")
        print(f"  Val dataset: {len(val_dataset)} samples")
        print(f"  Test dataset: {len(test_dataset)} samples")
        
        # Test sample loading
        print("🧪 Testing sample loading...")
        sample = train_dataset[0]
        print(f"  ✅ Video shape: {sample['pixel_values'].shape}")
        print(f"  ✅ Audio shape: {sample['audio_input_features'].shape} ...")
        
        model.eval()

        with torch.no_grad():
            batch = {}

            # ===== VIDEO =====
            batch["pixel_values"] = sample["pixel_values"].unsqueeze(0).to(device)

            # ===== AUDIO =====
            if "audio_input_features" in sample:
                audio = sample["audio_input_features"]  # (64, 1000)

                # reshape cho CLAP
                audio = audio.unsqueeze(0)              # (1, 64, 1000)
                audio = audio.permute(0, 2, 1)          # (1, 1000, 64)
                audio = audio.unsqueeze(1)              # (1, 1, 1000, 64)

                batch["audio_input_features"] = audio.to(device)
                batch["audio_is_longer"] = torch.tensor([False]).to(device)

            else:
                # fallback đúng shape CLAP
                batch["audio_input_features"] = torch.zeros(1, 1, 1000, 64).to(device)
                batch["audio_is_longer"] = torch.tensor([False]).to(device)

            # ===== LABEL (optional) =====
            if "labels" in sample:
                batch["labels"] = sample["labels"].unsqueeze(0).to(device)

            # ===== FORWARD =====
            output = model(**batch)

            print(f"✅ Logits shape: {output['logits'].shape}")
        
        # Training setup with FIXED batch settings
        training_args = TrainingArguments(
            output_dir=checkpoint_dir,
            num_train_epochs = num_train_epochs,
            per_device_train_batch_size=per_device_train_batch_size,
            per_device_eval_batch_size=per_device_eval_batch_size,
            gradient_accumulation_steps=gradient_accumulation_steps,
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            warmup_steps=warmup_steps,
            logging_steps=logging_steps,
            eval_strategy=eval_strategy,
            eval_steps=eval_steps,
            save_strategy=save_strategy,
            save_steps=save_steps,
            load_best_model_at_end=load_best_model_at_end,
            metric_for_best_model=metric_for_best_model,
            fp16=fp16,
            dataloader_num_workers=dataloader_num_workers,
            remove_unused_columns=remove_unused_columns,
            save_total_limit = save_total_limit,
            # FIXED: Don't drop last to preserve data count
            dataloader_drop_last=dataloader_drop_last,  # Changed from True to False
            max_grad_norm=max_grad_norm,
            disable_tqdm=disable_tqdm,
            report_to=report_to,
            logging_dir=logging_dir
        )
        
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            data_collator=smart_collate_fn,
            compute_metrics=compute_comprehensive_metrics,
        )
        
        # Start training
        print("\n" + "="*80)
        print("🚀 STARTING SMART ADAPTIVE TRAINING (FIXED)!")
        print("="*80)
        
        start_time = time.time()

        
        # Train with error suppression
        last_checkpoint = None

        if resume_training:
            if not os.path.isdir(checkpoint_dir):
                raise FileNotFoundError(f'❌ Can not found pretrained checkpoint at: {checkpoint_dir}')
            last_checkpoint = get_last_checkpoint(checkpoint_dir)
        
            if last_checkpoint is not None:
                print(f"🔁 Resuming from checkpoint: {last_checkpoint}")
                trainer.train(resume_from_checkpoint=True)
        else:
            print("Starting fresh training...")
            trainer.train()

        
        training_time = time.time() - start_time
        
        print(f"\n🎉 TRAINING COMPLETED in {training_time:.1f}s!")
        
        # Save best model
        trainer.save_model(best_checkpoint_dir)
        print("💾 Model saved!")
        
        
        # Evaluation
        print("📊 Running evaluation...")
        eval_results = trainer.evaluate()
        
        print(f"\n🎯 VALIDATION RESULTS:")
        print(f"  Accuracy: {eval_results['eval_accuracy']:.4f} ({eval_results['eval_accuracy']*100:.2f}%)")
        print(f"  F1-Score: {eval_results['eval_f1_score']:.4f}")
        print(f"  Recall: {eval_results['eval_recall']:.4f}")
        print(f"  Precision: {eval_results['eval_precision']:.4f}")
        
        # FIXED Test evaluation with manual prediction
        print("🧪 Running FIXED test evaluation...")
        
        # Use manual DataLoader to control exact sample count
        test_dataloader = DataLoader(
            test_dataset, 
            batch_size=8, 
            shuffle=False, 
            drop_last=False,  # Don't drop any samples
            collate_fn=smart_collate_fn
        )
        
        model.eval()
        all_predictions = []
        all_labels = []
        
        print(f"🔍 Processing test data in {len(test_dataloader)} batches...")
        
        with torch.no_grad():
            for batch_idx, batch in enumerate(test_dataloader):
                # Move to device
                batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
                
                # Get predictions
                outputs = model(**batch)
                predictions = torch.argmax(outputs['logits'], dim=1)
                
                all_predictions.extend(predictions.cpu().numpy())
                all_labels.extend(batch['labels'].cpu().numpy())
                
                if (batch_idx + 1) % 10 == 0:
                    print(f"  Processed batch {batch_idx + 1}/{len(test_dataloader)}")
        
        # Convert to numpy arrays
        y_pred = np.array(all_predictions)
        y_true = np.array(all_labels)
        
        print(f"\n🎯 FINAL TEST DATA VERIFICATION:")
        print(f"  Expected test samples: {len(test_dataset)}")
        print(f"  Actual predictions: {len(y_pred)}")
        print(f"  Actual labels: {len(y_true)}")
        print(f"  Data consistency: {'✅ CONSISTENT' if len(y_pred) == len(test_dataset) else '❌ INCONSISTENT'}")
        
        # Test metrics
        test_accuracy = accuracy_score(y_true, y_pred)
        test_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        test_recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
        test_precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
        
        print(f"\n🏆 FINAL TEST RESULTS (VERIFIED):")
        print(f"  Test samples: {len(y_true)}")
        print(f"  Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
        print(f"  F1-Score: {test_f1:.4f}")
        print(f"  Recall: {test_recall:.4f}")
        print(f"  Precision: {test_precision:.4f}")
        
        # Classification report
        print(f"\n📋 CLASSIFICATION REPORT (VERIFIED):")
        print("-" * 60)
        class_report = classification_report(
            y_true, y_pred, 
            target_names=data_manager.classes,
            digits=4
        )
        print(class_report)
        
        # Save results
        import json
        results = {
            'model_type': 'AUDIO-VIDEO_Multimodal_model',
            'data_verification': {
                'train_samples': len(train_dataset),
                'val_samples': len(val_dataset),
                'test_samples': len(test_dataset),
                'test_predictions': len(y_pred),
                'data_consistent': len(y_pred) == len(test_dataset)
            },
            'validation_metrics': {
                'accuracy': eval_results['eval_accuracy'],
                'f1_score': eval_results['eval_f1_score'],
                'recall': eval_results['eval_recall'],
                'precision': eval_results['eval_precision']
            },
            'test_metrics': {
                'accuracy': test_accuracy,
                'f1_score': test_f1,
                'recall': test_recall,
                'precision': test_precision
            },
            'training_time_seconds': training_time,
            'validation_stats': validator.stats,
            'dataset_sizes': {
                'train': len(train_dataset),
                'val': len(val_dataset),
                'test': len(test_dataset)
            }
        }
        
        with open('smart_results_fixed.json', 'w') as f:
            json.dump(results, f, indent=2)
        
        print("💾 Results saved!")
        
        # Final statistics
        print("\n📈 Final statistics:")
        train_dataset.get_stats()
        val_dataset.get_stats() 
        test_dataset.get_stats()
        
        print("\n" + "="*80)
        print("🎊 SMART TRAINING COMPLETED SUCCESSFULLY (FIXED)!")
        print("✅ DATA COUNTING ISSUES RESOLVED!")
        print("📊 ALL METRICS CALCULATED CORRECTLY!")
        print("="*80)

        
        # SAVE FULL HYPERPARAMETERS______________________________________________________________________
        full_config = {
            "training_args": trainer.args.to_dict(),
            "model_config": model.config.to_dict() if hasattr(model, "config") else {},
        }
        
        with open("/kaggle/working/experiment_config.json", "w") as f:
            json.dump(full_config, f, indent=4)


        # SAVE TRAINING HISTORY______________________________________________________________________
        with open("/kaggle/working/training_history.json", "w") as f:
            json.dump(trainer.state.log_history, f, indent=4)


        # SAVE FINAL METRICS______________________________________________________________________
        metrics = trainer.evaluate()
        
        with open("/kaggle/working/final_metrics.json", "w") as f:
            json.dump(metrics, f, indent=4)
        
        # SAVE ENVIRONMENT INFO______________________________________________________________________ 
        env_info = {
            "torch_version": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        }
        
        with open("/kaggle/working/environment_info.json", "w") as f:
            json.dump(env_info, f, indent=4)
        
        print("✅ Everything saved safely in /kaggle/working/")
        
    except Exception as e:
        print(f"❌ Unexpected error: {e}")
        import traceback
        traceback.print_exc()
    
    finally:
        cleanup_memory()

In [ ]:
train(
    # Data congig
    data_path = "/Users/nnam/Documents/Workspace/university/mguard/TikTok_Classification/datasets/dataset_with_relative_path.csv",
    data_mode='colab',
    
    # Video config
    max_frames=8, 
    image_size=224, 
     
    #Audio config
    target_sr=48000, 
    max_duration=10,
    
    # model config
    resume_training = False,
    video_model_name = "facebook/timesformer-base-finetuned-k400",
    audio_model_name = "laion/larger_clap_general",
    video_hidden_size = 768,
    audio_hidden_size = 512,
    fusion_hidden_size = 256,
    classifier_hidden_size = 128,
    dropout = 0.1,
    freeze_backbone=False,
    device='cuda',
    
    #Training config
    checkpoint_dir='',
    best_checkpoint_dir='',
    num_train_epochs = 20,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_score",
    fp16=True,
    dataloader_num_workers=0,
    remove_unused_columns=False,
    save_total_limit = 3,
    dataloader_drop_last=False,
    max_grad_norm=1.0,
    disable_tqdm=False,
    report_to="tensorboard",
    logging_dir="./runs"
)

In [ ]:
import os

checkpoint_dir = "/kaggle/working/checkpoints"
if os.path.exists(checkpoint_dir):
    files = os.listdir(checkpoint_dir)
    print(f"📁 Thư mục checkpoints chứa: {files}")
    if len(files) == 0:
        print("Empty! Đúng như dự đoán, chưa có checkpoint nào được lưu vì save_steps=100.")
else:
    print("❌ Thư mục checkpoints thậm chí còn chưa được tạo ra!")

### Inference

In [ ]:
import shutil
import subprocess
from pathlib import Path
def load_model(
    checkpoint_dir: str,
    device: Literal['cpu', 'cuda'] = "cuda"
):
    
    device = torch.device(device if torch.cuda.is_available() else "cpu")

    print("🔄 Loading config...")
    config = SmartMultimodalConfig.from_pretrained(checkpoint_dir)

    print("🧠 Initializing model...")
    model = SmartMultimodalModel(config)

    print("📦 Loading checkpoint...")
    state_dict = torch.load(f"{checkpoint_dir}/pytorch_model.bin", map_location=device)
    model.load_state_dict(state_dict)

    model.to(device)
    model.eval()

    print("✅ Model ready for inference!")
    return model
def extract_audio(
    video_path: str,
    output_path: str = None,
    ffmpeg_bin: str = "ffmpeg",
    bitrate: str = "192k",
    overwrite: bool = False,
):
    video_path = Path(video_path)

    # ===== OUTPUT PATH =====
    if output_path is None:
        output_path = video_path.with_suffix(".mp3")
    else:
        output_path = Path(output_path)

    # ===== SKIP IF EXISTS =====
    if output_path.exists() and not overwrite:
        print(f"⏭️ Skip: {output_path}")
        return str(output_path)

    # ===== TRY FFMPEG =====
    if shutil.which(ffmpeg_bin):
        try:
            print("🎧 Trying ffmpeg (copy)...")

            # Step 1: copy stream
            cmd_copy = [
                ffmpeg_bin,
                "-y" if overwrite else "-n",
                "-i", str(video_path),
                "-vn",
                "-c:a", "copy",
                str(output_path),
            ]

            result = subprocess.run(cmd_copy, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

            if result.returncode == 0:
                print("✅ Done with ffmpeg (copy)")
                return str(output_path)

            # Step 2: re-encode
            print("🔁 Fallback ffmpeg (encode)...")
            cmd_encode = [
                ffmpeg_bin,
                "-y" if overwrite else "-n",
                "-i", str(video_path),
                "-vn",
                "-c:a", "libmp3lame",
                "-b:a", bitrate,
                str(output_path),
            ]

            subprocess.run(cmd_encode, check=True)
            print("✅ Done with ffmpeg (encode)")
            return str(output_path)

        except Exception as e:
            print(f"⚠️ ffmpeg failed: {e}")

    # ===== FALLBACK MOVIEPY =====
    try:
        print("🎬 Trying moviepy...")
        from moviepy.editor import VideoFileClip

        with VideoFileClip(str(video_path)) as clip:
            if clip.audio is None:
                raise ValueError("No audio track found")

            clip.audio.write_audiofile(str(output_path), bitrate=bitrate, logger=None)

        print("✅ Done with moviepy")
        return str(output_path)

    except Exception as e:
        print(f"❌ All methods failed: {e}")
        return None
    
def inference(
    model,
    video_path: str,
    audio_path: str = None,
    label_map: dict = None,
    device: str = "cuda",
    max_frames: int = 8,
    image_size: int = 224,
    target_sr: int = 48000,
    max_duration: int = 10
):

    model.eval()
    device = torch.device(device if torch.cuda.is_available() else "cpu")
    model.to(device)

    # ===== LOAD DATA giống Dataset =====
    video_loader = RobustVideoLoader(max_frames=max_frames, image_size=image_size)
    audio_loader = RobustAudioLoader(target_sr=target_sr, max_duration=max_duration)

    sample = {}

    # ===== VIDEO =====
    pixel_values = video_loader.load(video_path)  # (T, 3, 224, 224)
    sample["pixel_values"] = pixel_values

    # ===== AUDIO =====
    if audio_path is not None:
        audio = audio_loader.load(audio_path)  # (64, 1000)
    else:
        audio = torch.zeros(64, 1000)

    sample["audio_input_features"] = audio

    # ===== BUILD BATCH =====
    batch = {}

    # VIDEO
    batch["pixel_values"] = sample["pixel_values"].unsqueeze(0).to(device)

    # AUDIO (QUAN TRỌNG)
    audio = sample["audio_input_features"]  # (64, 1000)

    audio = audio.unsqueeze(0)              # (1, 64, 1000)
    audio = audio.permute(0, 2, 1)          # (1, 1000, 64)
    audio = audio.unsqueeze(1)              # (1, 1, 1000, 64)

    batch["audio_input_features"] = audio.to(device)
    batch["audio_is_longer"] = torch.tensor([False]).to(device)

    # ===== FORWARD =====
    with torch.no_grad():
        outputs = model(**batch)
        logits = outputs["logits"]

        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()

    # ===== LABEL MAP =====
    if label_map:
        pred_label = label_map[pred]
    else:
        pred_label = pred

    return {
        "pred": pred,
        "label": pred_label,
        "probs": probs.cpu().numpy()[0]
    }

In [ ]:
video_path = ''
audio_path = ''
checkpoint_dir= ''
label_map={
    'Adult Content':0,
    'Harmful Content':1,
    'Safe': 2,
    'Suicide Content':3
}
extract_audio(video_path=video_path, output_path=audio_path)
model = load_model(checkpoint_dir=checkpoint_dir, device='cuda')
result = inference(
    model=model,
    video_path=video_path,
    audio_path= audio_path
)
print(f'============INFERENCE RESULT============')
print(result)

## Error analyst

Analyse with hashtags

In [ ]:
# ============================================================
# ERROR ANALYSIS MODULE
# ============================================================
import os, json, pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, f1_score
)
from torch.utils.data import DataLoader
import torch


# ─────────────────────────────────────────────
# 1. Load model từ final_model dir
# ─────────────────────────────────────────────
def load_model_for_analysis(
    final_model_dir: str,
    config: SmartMultimodalConfig,
    device: torch.device,
) -> SmartMultimodalModel:
    """Load best model từ final_model dir (do trainer.save_model() lưu)."""
    print(f"📂 Loading model from: {final_model_dir}")

    if not os.path.isdir(final_model_dir):
        raise FileNotFoundError(f"Final model dir not found: {final_model_dir}")

    model = SmartMultimodalModel(config)

    safe_path = os.path.join(final_model_dir, "model.safetensors")
    bin_path  = os.path.join(final_model_dir, "pytorch_model.bin")

    if os.path.exists(safe_path):
        from safetensors.torch import load_file
        state_dict = load_file(safe_path, device=str(device))
        print("  ✅ Loaded from model.safetensors")
    elif os.path.exists(bin_path):
        state_dict = torch.load(bin_path, map_location=device)
        print("  ✅ Loaded from pytorch_model.bin")
    else:
        raise FileNotFoundError(
            f"No weight file found in {final_model_dir}\n"
            f"  Expected: {safe_path}\n       or: {bin_path}"
        )

    model.load_state_dict(state_dict, strict=False)
    model.to(device)
    model.eval()
    print("✅ Model ready for inference")
    return model


# ─────────────────────────────────────────────
# 2. Collect predictions
# ─────────────────────────────────────────────
def collect_predictions(
    model: SmartMultimodalModel,
    dataset,
    dataframe: pd.DataFrame,
    device: torch.device,
    batch_size: int = 8,
    collate_fn=None,
) -> pd.DataFrame:
    """
    Chạy inference toàn bộ dataset, trả về DataFrame có thêm cột:
    true_idx, pred_idx, confidence, is_wrong, prob_vector
    """
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        collate_fn=collate_fn,
        num_workers=0,
    )

    all_preds, all_probs, all_labels = [], [], []

    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            batch = {
                k: v.to(device) if torch.is_tensor(v) else v
                for k, v in batch.items()
            }
            outputs = model(**batch)
            logits  = outputs["logits"]
            probs   = torch.softmax(logits, dim=-1).cpu().numpy()
            preds   = logits.argmax(dim=-1).cpu().numpy()
            labels  = batch["labels"].cpu().numpy()

            all_preds.extend(preds)
            all_probs.extend(probs)
            all_labels.extend(labels)

            if (batch_idx + 1) % 20 == 0:
                print(f"  Processed {batch_idx + 1}/{len(loader)} batches...")

    result_df = dataframe.copy().reset_index(drop=True)
    result_df["true_idx"]   = all_labels
    result_df["pred_idx"]   = all_preds
    result_df["confidence"] = [p[pred] for p, pred in zip(all_probs, all_preds)]
    result_df["is_wrong"]   = result_df["true_idx"] != result_df["pred_idx"]
    result_df["prob_vector"] = [json.dumps(p.tolist()) for p in all_probs]

    print(f"\n📊 Prediction summary:")
    print(f"  Total samples : {len(result_df)}")
    print(f"  Correct       : {(~result_df['is_wrong']).sum()}")
    print(f"  Wrong         : {result_df['is_wrong'].sum()}")
    print(f"  Accuracy      : {(~result_df['is_wrong']).mean()*100:.2f}%")
    return result_df


# ─────────────────────────────────────────────
# 3. Confusion matrix
# ─────────────────────────────────────────────
def plot_confusion_matrix(
    result_df: pd.DataFrame,
    class_names: list,
    save_dir: str,
) -> np.ndarray:
    y_true = result_df["true_idx"].values
    y_pred = result_df["pred_idx"].values

    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(
        1, 2,
        figsize=(max(12, len(class_names) * 1.2),
                 max(8,  len(class_names) * 0.9))
    )
    for ax, data, title, fmt in zip(
        axes,
        [cm, cm_norm],
        ["Confusion matrix (count)", "Confusion matrix (normalized)"],
        ["d", ".2f"],
    ):
        sns.heatmap(
            data, annot=True, fmt=fmt, cmap="Blues",
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.4, ax=ax,
        )
        ax.set_title(title, fontsize=12)
        ax.set_xlabel("Predicted", fontsize=10)
        ax.set_ylabel("True", fontsize=10)
        ax.tick_params(axis="x", rotation=45, labelsize=8)
        ax.tick_params(axis="y", rotation=0,  labelsize=8)

    plt.tight_layout()
    path = os.path.join(save_dir, "confusion_matrix.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"💾 Confusion matrix saved → {path}")

    # Per-class stats
    per_class = pd.DataFrame({
        "class":     class_names,
        "n_samples": cm.sum(axis=1),
        "n_correct": cm.diagonal(),
        "recall":    cm_norm.diagonal(),
    })
    per_class["error_rate"] = 1 - per_class["recall"]
    per_class = per_class.sort_values("error_rate", ascending=False)

    print("\n🔍 Per-class error rates (worst first):")
    print(per_class.to_string(index=False))
    per_class.to_csv(os.path.join(save_dir, "per_class_stats.csv"), index=False)

    return cm


# ─────────────────────────────────────────────
# 4. Lưu wrong samples
# ─────────────────────────────────────────────
def save_wrong_samples(
    result_df: pd.DataFrame,
    class_names: list,
    save_dir: str,
    top_k: int = 200,
) -> pd.DataFrame:
    idx2cls  = {i: c for i, c in enumerate(class_names)}
    wrong_df = result_df[result_df["is_wrong"]].copy()

    wrong_df["true_label"] = wrong_df["true_idx"].map(idx2cls)
    wrong_df["pred_label"] = wrong_df["pred_idx"].map(idx2cls)
    wrong_df["error_pair"] = wrong_df["true_label"] + " → " + wrong_df["pred_label"]

    # Model tự tin cao nhưng vẫn sai = nguy hiểm nhất → lên đầu
    wrong_df = wrong_df.sort_values("confidence", ascending=False)

    csv_path  = os.path.join(save_dir, "wrong_predictions.csv")
    json_path = os.path.join(save_dir, "wrong_predictions.json")

    wrong_df.drop(columns=["prob_vector"], errors="ignore").to_csv(csv_path, index=False)
    wrong_df.head(top_k).drop(columns=["prob_vector"], errors="ignore").to_json(
        json_path, orient="records", force_ascii=False, indent=2
    )
    print(f"💾 Wrong samples saved → {csv_path} ({len(wrong_df)} rows)")

    # Top confused pairs
    pair_counts = wrong_df["error_pair"].value_counts().head(20)
    print("\n🔴 Top confused class pairs:")
    print(pair_counts.to_string())
    pair_counts.to_csv(os.path.join(save_dir, "top_confused_pairs.csv"), header=["count"])

    # Bar chart top confused pairs
    fig, ax = plt.subplots(figsize=(10, max(4, len(pair_counts) * 0.4)))
    pair_counts[::-1].plot(kind="barh", ax=ax, color="salmon")
    ax.set_title("Top confused class pairs", fontsize=12)
    ax.set_xlabel("Count", fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "top_confused_pairs.png"), dpi=150, bbox_inches="tight")
    plt.close()

    return wrong_df


def run_error_analysis(
    final_model_dir: str = "/kaggle/working/final_model",
    analysis_dir:    str = "/kaggle/working/error_analysis",
    hashtag_pkl:     str = "path_to_your.pkl",
    hashtag_col:     str = "hashtags",
    batch_size:      int = 8,
    top_n:           int = 30,
):
    os.makedirs(analysis_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ── Load hashtag vectors ──
    print("📦 Loading hashtag vectors...")
    with open(hashtag_pkl, "rb") as f:
        pkl = pickle.load(f)
    hashtag_vector_map = {
        item.raw_hashtag.replace("#", ""): torch.tensor(
            item.cluster_vector, dtype=torch.float32
        )
        for item in pkl
    }
    print(f"  {len(hashtag_vector_map)} hashtag vectors loaded")

    # ── Load audio vectors ──
    audio_pkl = "/kaggle/input/datasets/ziaxxx/audio-embedding-clap/audio_embedding_clap.pkl"
    with open(audio_pkl, "rb") as f:
        pkl_audio = pickle.load(f)
    
    audio_embedding_map = {
        item.video_path: torch.tensor(item.embedding, dtype=torch.float32)
        for item in pkl_audio
    }
    print(f"  {len(audio_embedding_map)} audio embeddings loaded")
        
    # ── Load data ──
    print("\n📂 Loading dataset...")
    global data_manager
    data_manager = SmartDataManager("/kaggle/input/datasets/dnnamm/dataset-with-all-hashtag/dataset_with_all_hashtag.csv")
    data_manager.load_data()

    tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
    config    = SmartMultimodalConfig(
        num_classes     = data_manager.num_classes,
        # text_model_name = "bert-base-multilingual-cased",
        max_frames      = 8,
        image_size      = 224,
        dropout         = 0.1,
    )

    # ── Load best model từ final_model ──
    model = load_model_for_analysis(final_model_dir, config, device)

    # ── Build test split ──
    validator    = SmartVideoValidator(strict_mode=False)
    _, _, test_df = data_manager.get_smart_splits(validator)
    test_dataset = SmartDataset(
        test_df, audio_embedding_map, hashtag_vector_map, dataset_name="TEST"
    )
    print(f"\n📊 Test set: {len(test_dataset)} samples | {data_manager.num_classes} classes")

    # ── Collect predictions ──
    result_df = collect_predictions(
        model, test_dataset, test_df, device,
        batch_size=batch_size, collate_fn=smart_collate_fn,
    )

    # ── Confusion matrix ──
    print("\n🔢 Computing confusion matrix...")
    cm = plot_confusion_matrix(result_df, data_manager.classes, analysis_dir)

    # ── Wrong samples ──
    print("\n💾 Saving wrong predictions...")
    wrong_df = save_wrong_samples(result_df, data_manager.classes, analysis_dir)

    # # ── Hashtag error analysis ──
    # print("\n🏷️  Running hashtag error analysis...")
    # hashtag_error_analysis(
    #     result_df, data_manager.classes, analysis_dir,
    #     hashtag_col=hashtag_col, top_n=top_n,
    # )

    # ── Classification report ──
    y_true = result_df["true_idx"].values
    y_pred = result_df["pred_idx"].values
    print(f"\n📋 Classification report:")
    print("-" * 60)
    print(classification_report(y_true, y_pred, target_names=data_manager.classes, digits=4, zero_division=0))

    # ── Final summary JSON ──
    summary = {
        "final_model_dir": final_model_dir,
        "total_samples":   int(len(result_df)),
        "total_errors":    int(result_df["is_wrong"].sum()),
        "accuracy":        float(accuracy_score(y_true, y_pred)),
        "f1_macro":        float(f1_score(y_true, y_pred, average="macro",  zero_division=0)),
        "f1_weighted":     float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "classification_report": classification_report(
            y_true, y_pred,
            target_names=data_manager.classes,
            output_dict=True,
            zero_division=0,
        ),
    }
    summary_path = os.path.join(analysis_dir, "error_analysis_summary.json")
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    print("\n" + "=" * 60)
    print("✅ ERROR ANALYSIS COMPLETE")
    print(f"   Model       : {final_model_dir}")
    print(f"   Accuracy    : {summary['accuracy']*100:.2f}%")
    print(f"   F1 macro    : {summary['f1_macro']:.4f}")
    print(f"   F1 weighted : {summary['f1_weighted']:.4f}")
    print(f"   Errors      : {summary['total_errors']} / {summary['total_samples']}")
    print(f"   Outputs  →  {analysis_dir}/")
    print("=" * 60)

    return result_df, wrong_df, summary

In [ ]:
# ─────────────────────────────────────────────
# Entry point
# ─────────────────────────────────────────────
if __name__ == "__main__":
    result_df, wrong_df, summary = run_error_analysis(
        final_model_dir = "/kaggle/working/final_model",
        analysis_dir    = "/kaggle/working/error_analysis",
        hashtag_pkl     = "/kaggle/input/datasets/dnnamm/hashtag-classification-result/hashtag_classification_results.pkl",
        hashtag_col     = "hashtags",   # tên cột chứa list hashtag trong dataframe
        batch_size      = 8,
        top_n           = 30,           # số hashtag hiển thị trong chart/heatmap
    )

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/error_analysis",  # tên file zip
    'zip',
    "/kaggle/working/error_analysis"   # folder cần zip
)